In [1]:
from functools import partial
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
from torchvision import datasets,transforms
from torchvision.transforms import ToTensor

In [2]:
# Import numpy for math
import numpy as np

# Import matplotlib for visualization
import matplotlib.pyplot as plt

# Check versions
# Note: your PyTorch version shouldn't be lower than 1.10.0 and torchvision version shouldn't be lower than 0.11
print(f"PyTorch version: {torch.__version__}\ntorchvision version: {torchvision.__version__}")

PyTorch version: 2.9.1+cu130
torchvision version: 0.24.1+cu130


In [34]:
torch.cuda.is_available()

True

### DATA LOADING

In [6]:
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
from torch.utils.data import random_split

images_dir = "../dataset/images_dataset"

# Load without transforms for now
images_dataset = datasets.ImageFolder(root=images_dir, transform=None)

# Print classes
print("Classes found:", images_dataset.classes)
print("Class-to-index mapping:", images_dataset.class_to_idx)

# Split dataset
total_size = len(images_dataset)
val_size = int(0.20 * total_size)
train_size = total_size - val_size

train_dataset, val_dataset = random_split(images_dataset, [train_size, val_size])

print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))

# ----------- TRANSFORMS -----------
train_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])

val_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor()
])

# Apply transforms to the underlying ImageFolder
train_dataset.dataset.transform = train_transform
val_dataset.dataset.transform = val_transform


# ----------- DATALOADERS -----------
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

Classes found: ['forgetable', 'memorable']
Class-to-index mapping: {'forgetable': 0, 'memorable': 1}
Train size: 5733
Val size: 1433


### AUTO RESOLUTION SEARCH

In [7]:
RESOLUTIONS = [224, 260, 300, 384,512, 650]  # adapt as needed
EPOCHS_PER_RES = 2                  # fast test

In [8]:
import time
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
import torch.nn as nn

def auto_search_resolution(model_builder, train_dataset, val_dataset, resolutions, epochs, batch_size, device):
    results = {}

    for resolution in resolutions:
        print("\n" + "="*80)
        print(f"🔍 TESTING RESOLUTION: {resolution} x {resolution}")
        print("="*80)

        # APPLY TRANSFORMS
        transform = transforms.Compose([
            transforms.Resize((resolution, resolution)),
            transforms.ToTensor(),
        ])
        train_dataset.dataset.transform = transform
        val_dataset.dataset.transform = transform

        # DataLoaders
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        # Model
        model = model_builder(weights=None).to(device)
        model.fc = nn.Linear(model.fc.in_features, 2).to(device)

        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        criterion = nn.CrossEntropyLoss()

        print(f"🧠 Model initialized at resolution {resolution}.")
        print(f"📦 Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

        # Epoch loop
        for epoch in range(epochs):
            epoch_start = time.time()
            model.train()
            running_loss = 0
            batch_count = len(train_loader)

            print(f"\n--- Epoch {epoch+1}/{epochs} ---")

            for batch_idx, (x, y) in enumerate(train_loader):
                x, y = x.to(device), y.to(device)

                optimizer.zero_grad()
                loss = criterion(model(x), y)
                loss.backward()
                optimizer.step()

                running_loss += loss.item()

                if batch_idx % 10 == 0:
                    used = torch.cuda.memory_allocated(device) / (1024**2)
                    print(f"  Batch {batch_idx+1}/{batch_count} | "
                          f"Loss: {loss.item():.4f} | GPU: {used:.1f} MB")

            # Validation
            model.eval()
            val_loss = 0
            with torch.no_grad():
                for x, y in val_loader:
                    x, y = x.to(device), y.to(device)
                    val_loss += criterion(model(x), y).item()

            val_loss /= len(val_loader)
            results[resolution] = val_loss

            epoch_time = time.time() - epoch_start
            print(f"✓ Epoch {epoch+1} completed in {epoch_time:.1f}s")
            print(f"   🔹 Train Loss: {running_loss / batch_count:.4f}")
            print(f"   🔹 Val Loss:   {val_loss:.4f}")

        print(f"🏁 Finished resolution {resolution} → Final Val Loss: {val_loss:.4f}")

    # Best resolution
    best_res = min(results, key=results.get)
    print("\n" + "="*80)
    print(f"🏆 BEST RESOLUTION: {best_res} (val loss = {results[best_res]:.4f})")
    print("="*80)

    return best_res, results


In [ ]:
from torchvision.models import resnet50

best_resolution, all_results = auto_search_resolution(
    model_builder=resnet50,
    train_dataset= train_dataset,
    val_dataset= val_dataset,
    resolutions=RESOLUTIONS,
    epochs=EPOCHS_PER_RES,
    batch_size=32,
    device="cuda"
)


🔍 TESTING RESOLUTION: 224 x 224
🧠 Model initialized at resolution 224.
📦 Train batches: 180, Val batches: 45

--- Epoch 1/2 ---
  Batch 1/180 | Loss: 0.8446 | GPU: 402.7 MB
  Batch 11/180 | Loss: 0.7091 | GPU: 405.6 MB
  Batch 21/180 | Loss: 0.5407 | GPU: 405.6 MB
  Batch 31/180 | Loss: 0.5155 | GPU: 405.6 MB
  Batch 41/180 | Loss: 0.5240 | GPU: 405.6 MB
  Batch 51/180 | Loss: 0.4769 | GPU: 405.6 MB
  Batch 61/180 | Loss: 0.4853 | GPU: 405.6 MB
  Batch 71/180 | Loss: 0.5938 | GPU: 405.6 MB
  Batch 81/180 | Loss: 0.6603 | GPU: 405.6 MB
  Batch 91/180 | Loss: 0.4314 | GPU: 405.6 MB
  Batch 101/180 | Loss: 0.5186 | GPU: 405.6 MB
  Batch 111/180 | Loss: 0.4431 | GPU: 405.6 MB
  Batch 121/180 | Loss: 0.4673 | GPU: 405.6 MB
  Batch 131/180 | Loss: 0.5636 | GPU: 405.6 MB
  Batch 141/180 | Loss: 0.5300 | GPU: 405.6 MB
  Batch 151/180 | Loss: 0.4702 | GPU: 405.6 MB
  Batch 161/180 | Loss: 0.5445 | GPU: 405.6 MB
  Batch 171/180 | Loss: 0.4550 | GPU: 405.6 MB
✓ Epoch 1 completed in 198.9s
   🔹 T